# Pipeline Check
**Purpose:** quick checks that the datamart produced by the pipeline is correct and ML-ready.
1. Gold feature store and label store: row counts, schema consistency across months, nulls
2. Join features to labels and check bad rate per month
3. Simple sanity model with a time-based split (no leakage)

Run from the `assignment_2/` folder after the pipeline has created `datamart/`.

In [ ]:
# notebooks live in assignment_2/notebooks/: run from assignment_2/ so relative paths
# (datamart/, model_bank/, utils/) work, same as the Airflow scripts
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("working directory:", os.getcwd())

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

## 1. Load gold tables

In [2]:
gold_feature_store_directory = "datamart/gold/feature_store/"
gold_label_store_directory = "datamart/gold/label_store/"

feature_files = sorted(glob.glob(os.path.join(gold_feature_store_directory, "*.parquet")))
label_files = sorted(glob.glob(os.path.join(gold_label_store_directory, "*.parquet")))

df_features = pd.concat([pd.read_parquet(f) for f in feature_files])
df_labels = pd.concat([pd.read_parquet(f) for f in label_files])

print("feature partitions:", len(feature_files), "| rows:", len(df_features), "| cols:", df_features.shape[1])
print("label partitions:  ", len(label_files), "| rows:", len(df_labels))

feature partitions: 25 | rows: 12500 | cols: 65
label partitions:   35 | rows: 12500


In [3]:
# every monthly partition should have the same columns (no schema drift)
schemas = {str(pq.read_table(f).schema) for f in feature_files}
print("distinct feature store schemas:", len(schemas))

# rows per month
df_features.groupby("snapshot_date").size().to_frame("new_loans").T

distinct feature store schemas: 1


snapshot_date,2023-01-01,2023-02-01,2023-03-01,2023-04-01,2023-05-01,2023-06-01,2023-07-01,2023-08-01,2023-09-01,2023-10-01,...,2024-04-01,2024-05-01,2024-06-01,2024-07-01,2024-08-01,2024-09-01,2024-10-01,2024-11-01,2024-12-01,2025-01-01
new_loans,530,501,506,510,521,517,471,481,454,487,...,513,491,498,505,543,493,456,488,515,526


In [4]:
# null rate per column (top 15)
df_features.isnull().mean().sort_values(ascending=False).head(15).round(3)

fe_7_mean     0.282
fe_5_mean     0.282
fe_8_mean     0.282
fe_9_mean     0.282
fe_10_mean    0.282
fe_11_mean    0.282
fe_12_mean    0.282
fe_13_mean    0.282
fe_14_mean    0.282
fe_15_mean    0.282
fe_16_mean    0.282
fe_2_mean     0.282
fe_1_mean     0.282
fe_18_mean    0.282
fe_17_mean    0.282
dtype: float64

## 2. Join features and labels
Feature snapshot = loan application month; label snapshot = application + 6 months (30dpd_6mob). Join on `loan_id`.

In [5]:
df = df_features.merge(
    df_labels[["loan_id", "label", "snapshot_date"]].rename(columns={"snapshot_date": "label_snapshot_date"}),
    on="loan_id",
    how="inner",
)
print("joined rows:", len(df))

# bad rate by application month
df.groupby("snapshot_date")["label"].mean().round(3).to_frame("bad_rate").T

joined rows: 12500


snapshot_date,2023-01-01,2023-02-01,2023-03-01,2023-04-01,2023-05-01,2023-06-01,2023-07-01,2023-08-01,2023-09-01,2023-10-01,...,2024-04-01,2024-05-01,2024-06-01,2024-07-01,2024-08-01,2024-09-01,2024-10-01,2024-11-01,2024-12-01,2025-01-01
bad_rate,0.264,0.307,0.32,0.269,0.267,0.265,0.297,0.289,0.262,0.257,...,0.277,0.303,0.311,0.307,0.3,0.331,0.246,0.281,0.272,0.268


## 3. Sanity model (time-based split)
Train on earlier application months, test on later months. Imputer and scaler are fitted on train only.

In [6]:
drop_cols = ["Customer_ID", "loan_id", "snapshot_date", "label_snapshot_date", "label"]
feature_cols = [c for c in df.columns if c not in drop_cols]

train_cutoff = pd.Timestamp("2024-06-01").date()
train = df[df["snapshot_date"] <= train_cutoff]
test = df[df["snapshot_date"] > train_cutoff]
print("train:", len(train), "| test:", len(test), "| features:", len(feature_cols))

model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000))
model.fit(train[feature_cols], train["label"])

auc = roc_auc_score(test["label"], model.predict_proba(test[feature_cols])[:, 1])
print(f"test AUC: {auc:.4f} | Gini: {2*auc-1:.4f}")

train: 8974 | test: 3526 | features: 62


test AUC: 0.7719 | Gini: 0.5438


Conclusion:
- Feature store is ML-compatible (numeric, consistent schema, joinable to labels)
- AUC is reasonable and not suspiciously close to 1.0, so there is no obvious leakage

## 4. Model bank / predictions / monitoring
To be added after the Airflow DAG produces `model_bank/`, `datamart/gold/model_predictions/` and `datamart/gold/model_monitoring/`.